<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/05-data-sources-io/02-partitionby-pushdown-pruning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 5.2 — partitionBy, predicate pushdown, column pruning

Every claim in the lesson gets verified against `explain()` output here.
Writes to local `output/`, cleaned up at the end.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("5.2-partition-pushdown-pruning")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True).schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)

## Write partitioned, inspect the directory tree

In [ ]:
(orders.write
 .partitionBy("country")
 .mode("overwrite")
 .parquet("output/orders_by_country"))

import pathlib
for d in sorted(pathlib.Path("output/orders_by_country").iterdir()):
    if d.is_dir():
        files = [f for f in d.iterdir() if f.suffix == ".parquet"]
        print(f"{d.name:<40} {len(files)} file(s)")
# note the __HIVE_DEFAULT_PARTITION__ directory: our 3 null-country rows

In [ ]:
# The country column moved OUT of the files, INTO the paths — proof:
one_dir = spark.read.parquet("output/orders_by_country/country=IN")
print("columns reading ONE directory:", "country" in one_dir.columns)

whole = spark.read.parquet("output/orders_by_country")
print("columns reading the ROOT:     ", "country" in whole.columns, "(reconstructed from paths)")

## Partition pruning: find PartitionFilters in the plan

In [ ]:
pruned = whole.filter(col("country") == "IN")
pruned.explain()
# scan node: PartitionFilters: [isnotnull(country), (country = IN)]
# -> only the country=IN directory is ever opened

## Predicate pushdown: PushedFilters — and what refuses to push

In [ ]:
# Pushes: comparison on a raw stored column
whole.filter(col("quantity") >= 3).explain()
# scan node: PushedFilters: [IsNotNull(quantity), GreaterThanOrEqual(quantity,3)]

In [ ]:
# Does NOT push: expression wrapped around the column
whole.filter(F.year(col("order_date")) == 2026).explain()
# PushedFilters: [] for the year() — the filter runs AFTER a full read

# The rewrite that pushes: raw-column range
whole.filter(
    (col("order_date") >= "2026-01-01") & (col("order_date") < "2027-01-01")
).explain()

## Column pruning: ReadSchema shrinks to what you select

In [ ]:
whole.select("order_id", "quantity").explain()
# ReadSchema: struct<order_id:int,quantity:int> — two columns' bytes, not eight

## The file-explosion interaction — and the repartition fix

In [ ]:
def count_files(root):
    return sum(1 for p in pathlib.Path(root).rglob("*.parquet"))

# 8 in-memory partitions x up to 5 country dirs = many small files
orders.repartition(8).write.partitionBy("country").mode("overwrite").parquet("output/explode")
print("naive:", count_files("output/explode"), "files")

# align memory layout with disk layout first -> one file per directory
orders.repartition("country").write.partitionBy("country").mode("overwrite").parquet("output/aligned")
print("aligned:", count_files("output/aligned"), "files")

## Cleanup

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)
print("cleaned up.")

## Exercises

1. Write orders partitioned by (`category`, `country`) — how many directories? Read back with a filter on ONLY `country` — does the plan still show partition pruning on the second level?
2. Filter the partitioned dataset with `col("country").isin("IN","UK")` and with `col("country") != "IN"` — check PartitionFilters for each. Does negation prune?
3. Generate 1M rows with a sorted `id` column, write parquet, and filter `id BETWEEN 100 AND 200`; then repeat with the rows shuffled (`orderBy(F.rand())`) before writing. Compare the two runs in the SQL tab ('number of files read' / bytes) — the sorted-layout effect on row-group stats.
4. What happens if you `partitionBy("order_id")` here (41 unique values)? Count the directories, imagine 41 million, and write the lesson's cardinality rule from memory.

In [ ]:
# your exercise code here
